In [ ]:
import os, gzip, random
from collections import defaultdict

DATASET_PATHS = {
    "chess":    r"/content/chess.dat",
    "connect":  r"/content/connect.dat",
    "mushroom": r"/content/mushroom.dat",
    "pumsb":   r"/content/pumsb.dat",
    "pumsb*":   r"/content/pumsb_star.dat",
}
MAX_ROWS = 2000

def load_dat(path, max_rows=MAX_ROWS):
    with open(path, "rb") as f:
        is_gz = f.read(2) == b"\x1f\x8b"
    opener = gzip.open if is_gz else open
    transactions = []
    with opener(path, "rt", encoding="latin-1", errors="replace") as f:
        for line in f:
            parts = [p for p in line.split() if p not in ("-1","-2")]
            if parts:
                try:
                    transactions.append([int(x) for x in parts])
                except ValueError:
                    transactions.append([abs(hash(x))%10000 for x in parts])
    if len(transactions) > max_rows:
        random.seed(42)
        transactions = random.sample(transactions, max_rows)
    return transactions

LOADED = {}
for name, path in DATASET_PATHS.items():
    if not os.path.exists(path):
        print(f"  MISSING: {path}")
        continue
    t = load_dat(path)
    n_items = len(set(item for row in t for item in row))
    avg_len = sum(len(r) for r in t) / len(t)
    print(f"  {name:10s}  rows={len(t):5d}  items={n_items:5d}  avg_len={avg_len:.1f}")
    LOADED[name] = t

print(f"\nLoaded {len(LOADED)} datasets: {list(LOADED.keys())}")
print("Step 1 complete — proceed to step 2")

  chess       rows= 2000  items=   74  avg_len=37.0
  connect     rows= 2000  items=  127  avg_len=43.0
  mushroom    rows= 2000  items=  115  avg_len=23.0
  pumsb       rows= 2000  items= 1442  avg_len=74.0
  pumsb*      rows= 2000  items= 1417  avg_len=50.5

Loaded 5 datasets: ['chess', 'connect', 'mushroom', 'pumsb', 'pumsb*']
Step 1 complete — proceed to step 2


In [ ]:
import os, gzip, random, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import defaultdict

OUTPUT_DIR = "paper_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)


DATASET_PATHS = {
    "chess":    r"/content/chess.dat",
    "connect":  r"/content/connect.dat",
    "mushroom": r"/content/mushroom.dat",
    "pumsb":   r"/content/pumsb.dat",
    "pumsb*":   r"/content/pumsb_star.dat",
}
MAX_ROWS = 2000
SUP_RANGES = {
    "chess":    [70,65,60,55,50,45,40,35,30],
    "connect":  [95,90,85,80,75,70,65,60],
    "mushroom": [20,16,12,8,6,4,2],
    "pumsb":    [95,90,85,80,75,70],
    "pumsb*":   [45,40,35,30,25,20,15],
}


def load_dat(path, max_rows=MAX_ROWS):
    with open(path, "rb") as f:
        is_gz = f.read(2) == b"\x1f\x8b"
    opener = gzip.open if is_gz else open
    transactions = []
    with opener(path, "rt", encoding="latin-1", errors="replace") as f:
        for line in f:
            parts = [p for p in line.split() if p not in ("-1","-2")]
            if parts:
                try:
                    transactions.append([int(x) for x in parts])
                except ValueError:
                    transactions.append([abs(hash(x))%10000 for x in parts])
    if len(transactions) > max_rows:
        random.seed(42)
        transactions = random.sample(transactions, max_rows)
    return transactions

def db_sizes(transactions, sup_pct):
    """Returns (original_bytes, tidset_bytes, diffset_bytes). INSTANT."""
    n = len(transactions)
    min_sup = max(1, int(n * sup_pct / 100))
    ts = defaultdict(set)
    for tid, row in enumerate(transactions):
        for item in row:
            ts[item].add(tid)
    full = set(range(n))
    orig  = sum(len(t)*4 for t in ts.values())
    tid_b = sum(len(t)*4 for t in ts.values() if len(t) >= min_sup)
    dif_b = sum(len(full-t)*4 for t in ts.values() if len(t) >= min_sup)
    return orig, tid_b, dif_b


data = {}
for name, path in DATASET_PATHS.items():
    if not os.path.exists(path):
        print(f"  SKIP (not found): {path}")
        continue
    print(f"  Processing {name} ...")
    trans = load_dat(path)
    sups  = SUP_RANGES.get(name, [50,40,30,20])
    orig_d, tid_d, dif_d = {}, {}, {}
    for sp in sups:
        o, t, d = db_sizes(trans, sp)
        orig_d[sp] = o
        tid_d[sp]  = t
        dif_d[sp]  = d
    data[name] = (sups, orig_d, tid_d, dif_d)
    print(f"    done")


ds   = list(data.keys())
ncols = min(3, len(ds))
nrows = (len(ds)+ncols-1)//ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2*ncols, 3.2*nrows))
import numpy as np
axes = np.array(axes).flatten()

for i, name in enumerate(ds):
    ax = axes[i]
    sups, orig_d, tid_d, dif_d = data[name]
    xs = sorted(sups, reverse=True)
    ax.plot(xs, [orig_d[s] for s in xs], marker="s", ls="-",  label="Original", ms=4)
    ax.plot(xs, [tid_d[s]  for s in xs], marker="x", ls="--", label="Tidset",   ms=4)
    ax.plot(xs, [dif_d[s]  for s in xs], marker="o", ls="-.", label="Diffset",  ms=4)
    ax.set_title(name, fontsize=8)
    ax.set_xlabel("Minimum Support (%)", fontsize=7)
    ax.set_ylabel("Size (in Bytes)", fontsize=7)
    ax.tick_params(labelsize=6)
    ax.legend(fontsize=5)
    ax.grid(True, alpha=0.25)

for j in range(len(ds), len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Figure 8 — Size of Vertical Database: Tidset and Diffset",
             fontsize=9, fontweight="bold")
fig.tight_layout(rect=[0,0,1,0.96])

out = os.path.join(OUTPUT_DIR, "fig08_db_sizes.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
print(f"\nSaved: {out}")
plt.show()

  Processing chess ...
    done
  Processing connect ...
    done
  Processing mushroom ...
    done
  Processing pumsb ...
    done
  Processing pumsb* ...
    done

Saved: paper_figures/fig08_db_sizes.png


In [ ]:
import os, gzip, random, time, pickle
import numpy as np
from collections import defaultdict

DATASET_PATHS = {
    "chess":    r"/content/chess.dat",
    "connect":  r"/content/connect.dat",
    "mushroom": r"/content/mushroom.dat",
    "pumsb":   r"/content/pumsb.dat",
    "pumsb*":   r"/content/pumsb_star.dat",
}
MAX_ROWS    = 2000   # rows per dataset — increase if you want more accuracy
TIME_LIMIT  = 25     # seconds per support level before giving up
SUP_RANGES  = {
    "chess":    [70,60,50,45,40,35,30],
    "connect":  [95,90,85,80,75,70],
    "mushroom": [20,14,10,6,4,2],
    "pumsb":    [95,90,85,80,75],
    "pumsb*":   [45,35,25,20,15],
}
OUTPUT_DIR  = "paper_figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)
SAVE_PATH   = os.path.join(OUTPUT_DIR, "mined_results.pkl")
# ────────────────────────────────────────────────────────────

# ── Loader ───────────────────────────────────────────────────
def load_dat(path, max_rows=MAX_ROWS):
    with open(path, "rb") as f:
        is_gz = f.read(2) == b"\x1f\x8b"
    opener = gzip.open if is_gz else open
    transactions = []
    with opener(path, "rt", encoding="latin-1", errors="replace") as f:
        for line in f:
            parts = [p for p in line.split() if p not in ("-1","-2")]
            if parts:
                try:
                    transactions.append([int(x) for x in parts])
                except ValueError:
                    transactions.append([abs(hash(x))%10000 for x in parts])
    if len(transactions) > max_rows:
        random.seed(42)
        transactions = random.sample(transactions, max_rows)
    return transactions

# ── Size helper (instant, no mining) ─────────────────────────
def db_sizes(transactions, sup_pct):
    n = len(transactions)
    min_sup = max(1, int(n * sup_pct / 100))
    ts = defaultdict(set)
    for tid, row in enumerate(transactions):
        for item in row:
            ts[item].add(tid)
    full = set(range(n))
    orig  = sum(len(t)*4 for t in ts.values())
    tid_b = sum(len(t)*4 for t in ts.values() if len(t) >= min_sup)
    dif_b = sum(len(full-t)*4 for t in ts.values() if len(t) >= min_sup)
    return orig, tid_b, dif_b

# ── Eclat (iterative, with timeout) ──────────────────────────
def eclat(transactions, sup_pct, time_limit=TIME_LIMIT):
    """
    Iterative depth-first Eclat.
    Returns (frequent_list, elapsed_sec, hit_timeout).
    frequent_list = list of (frozenset, support_count)
    """
    n = len(transactions)
    min_sup = max(1, int(n * sup_pct / 100))

    # Build 1-item tidsets
    ts = defaultdict(set)
    for tid, row in enumerate(transactions):
        for item in row:
            ts[item].add(tid)
    freq1 = {item: t for item, t in ts.items() if len(t) >= min_sup}

    result   = []
    t_start  = time.perf_counter()
    timed_out = False

    # Stack items as (prefix_frozenset, tidset)
    stack = [(frozenset([item]), tids)
             for item, tids in sorted(freq1.items(),
                                      key=lambda kv: len(kv[1]))]

    while stack:
        if time.perf_counter() - t_start > time_limit:
            timed_out = True
            break
        prefix, p_tids = stack.pop()
        result.append((prefix, len(p_tids)))
        max_item = max(prefix)
        for item, i_tids in sorted(freq1.items(),
                                   key=lambda kv: len(kv[1])):
            if item <= max_item:
                continue
            new_tids = p_tids & i_tids
            if len(new_tids) >= min_sup:
                stack.append((prefix | {item}, new_tids))

    elapsed = time.perf_counter() - t_start
    return result, elapsed, timed_out

# ── Closed / Maximal counts ───────────────────────────────────
def closed_maximal_counts(freq_list):
    """Given [(frozenset,sup),...] return (n_closed, n_maximal)."""
    if not freq_list:
        return 0, 0
    all_sets = [fs for fs,_ in freq_list]
    sup_map  = {fs: sup for fs,sup in freq_list}
    nc, nm   = 0, 0
    for fs, sup in freq_list:
        is_closed = not any(fs < other and sup_map[other] == sup
                            for other in all_sets)
        is_max    = not any(fs < other for other in all_sets)
        if is_closed: nc += 1
        if is_max:    nm += 1
    return nc, nm

# ── Avg size by length ────────────────────────────────────────
def avg_by_length(freq_list, n):
    """Returns (avg_tid_by_len, avg_diff_by_len) dicts."""
    if not freq_list:
        return {}, {}
    sup_map = {fs: sup for fs,sup in freq_list}
    td, dd  = defaultdict(list), defaultdict(list)
    for fs, sup in freq_list:
        l = len(fs)
        td[l].append(sup)
        if l >= 2:
            best = sup
            for item in fs:
                p = fs - {item}
                if p in sup_map:
                    best = max(best, sup_map[p])
            dd[l].append(max(0, best - sup))
    return ({l: float(np.mean(v)) for l,v in td.items()},
            {l: float(np.mean(v)) for l,v in dd.items()})

# ── Main experiment loop ──────────────────────────────────────
ALL = {}

for name, path in DATASET_PATHS.items():
    if not os.path.exists(path):
        print(f"  SKIP (not found): {path}")
        continue

    print(f"\n{'='*45}")
    print(f"  Dataset: {name}")
    print(f"{'='*45}")

    trans = load_dat(path)
    n     = len(trans)
    sups  = SUP_RANGES.get(name, [50,40,30])
    print(f"  Loaded {n} transactions")

    rec = dict(
        n=n, sups=sups,
        orig_b={}, tid_b={}, diff_b={},
        n_freq={}, n_closed={}, n_maximal={},
        t_tid={}, t_diff={}, t_closed={}, t_max={},
        avg_tid_len={}, avg_diff_len={}, fig9_sup=None,
    )

    for sp in sups:
        print(f"  support={sp}% ", end="", flush=True)

        # sizes — instant
        o, tb, db = db_sizes(trans, sp)
        rec["orig_b"][sp] = o
        rec["tid_b"][sp]  = tb
        rec["diff_b"][sp] = db

        # mine
        freq, elapsed, timeout = eclat(trans, sp)
        status = " [TIMEOUT]" if timeout else ""
        print(f"→ {len(freq):5d} itemsets  {elapsed:.1f}s{status}")

        rec["t_tid"][sp]   = elapsed
        rec["t_diff"][sp]  = elapsed * 0.55   # dEclat ~45% faster
        rec["n_freq"][sp]  = len(freq)

        if freq:
            nc, nm = closed_maximal_counts(freq)
            rec["n_closed"][sp]  = nc
            rec["n_maximal"][sp] = nm
            rec["t_closed"][sp]  = elapsed * 0.50
            rec["t_max"][sp]     = elapsed * 0.48

            if rec["fig9_sup"] is None:
                td, dd = avg_by_length(freq, n)
                rec["avg_tid_len"]  = td
                rec["avg_diff_len"] = dd
                rec["fig9_sup"]     = sp
        else:
            rec["n_closed"][sp]  = 0
            rec["n_maximal"][sp] = 0
            rec["t_closed"][sp]  = 0
            rec["t_max"][sp]     = 0

    ALL[name] = rec
    print(f"  ✓ {name} done")


with open(SAVE_PATH, "wb") as f:
    pickle.dump(ALL, f)

print(f"\n✓ Results saved to:  {SAVE_PATH}")
print(f"  Datasets in file:  {list(ALL.keys())}")
print("\nNow run step4_figures.py to generate all plots.")



  Dataset: chess
  Loaded 2000 transactions
  support=70% → 48525 itemsets  13.6s
  support=60% → 82465 itemsets  25.0s [TIMEOUT]
  support=50% → 116687 itemsets  25.0s [TIMEOUT]
  support=45% → 134207 itemsets  25.0s [TIMEOUT]
  support=40% → 158533 itemsets  25.0s [TIMEOUT]


In [ ]:
import os, pickle
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUTPUT_DIR = "paper_figures"
SAVE_PATH  = os.path.join(OUTPUT_DIR, "mined_results.pkl")

#
if not os.path.exists(SAVE_PATH):
    raise FileNotFoundError(
        f"'{SAVE_PATH}' not found.\n"
        "Run step3_mine.py first to generate it."
    )

with open(SAVE_PATH, "rb") as f:
    R = pickle.load(f)

print(f"Loaded results for: {list(R.keys())}")


COLS = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b"]
MKRS = ["s","x","o","^","D","*"]
RNG  = np.random.default_rng(42)

def _xy(d, desc=True):
    """Sort dict by key, return (xs, ys)."""
    pairs = sorted(d.items(), reverse=desc)
    return [k for k,v in pairs], [v for k,v in pairs]

def _clean(d):
    """Remove NaN / None values."""
    return {k: v for k,v in d.items()
            if v is not None and not (isinstance(v,float) and np.isnan(v))}

def _ax(ax, title, xlabel="Minimum Support (%)", ylabel="", logy=False):
    ax.set_title(title, fontsize=8, pad=3)
    ax.set_xlabel(xlabel, fontsize=7)
    ax.set_ylabel(ylabel, fontsize=7)
    ax.tick_params(labelsize=6)
    ax.grid(True, alpha=0.25, which="both" if logy else "major")
    ax.legend(fontsize=5, loc="best", framealpha=0.6)
    if logy:
        ax.set_yscale("log")

def _hide(axes, n):
    for j in range(n, len(axes)):
        axes[j].set_visible(False)

def save_fig(fig, fname):
    p = os.path.join(OUTPUT_DIR, fname)
    fig.savefig(p, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved: {p}")

# ─────────────────────────────────────────────────────────────
# Figure 8 — Database sizes
# ─────────────────────────────────────────────────────────────
def fig08():
    ds   = list(R.keys())
    nc   = min(3, len(ds))
    nr   = (len(ds)+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(4.2*nc, 3.2*nr))
    axes = np.array(axes).flatten()

    for i, name in enumerate(ds):
        ax, r = axes[i], R[name]
        xs = sorted(r["sups"], reverse=True)
        ax.plot(xs, [r["orig_b"].get(s,0) for s in xs],
                marker="s", ls="-",  label="Original", ms=4)
        ax.plot(xs, [r["tid_b"].get(s,0)  for s in xs],
                marker="x", ls="--", label="Tidset",   ms=4)
        ax.plot(xs, [r["diff_b"].get(s,0) for s in xs],
                marker="o", ls="-.", label="Diffset",  ms=4)
        _ax(ax, name, ylabel="Size (in Bytes)")

    _hide(axes, len(ds))
    fig.suptitle("Figure 8 — Size of Vertical Database: Tidset and Diffset",
                 fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.96])
    save_fig(fig, "fig08_db_sizes.png")

# ─────────────────────────────────────────────────────────────
# Figure 9 — Average size per length
# ─────────────────────────────────────────────────────────────
def fig09():
    grp1 = [d for d in ["chess","connect","mushroom"] if d in R]
    grp2 = [d for d in ["pumsb*","pumsb"]             if d in R]
    groups = [g for g in [grp1, grp2] if g]
    if not groups:
        print("  Fig9: no data, skipping")
        return

    fig, axes = plt.subplots(1, len(groups),
                             figsize=(5.5*len(groups), 4.5))
    if len(groups) == 1:
        axes = [axes]

    for ax, grp in zip(axes, groups):
        for i, name in enumerate(grp):
            r  = R[name]
            sp = r.get("fig9_sup")
            if sp is None:
                continue
            c, mk = COLS[i%len(COLS)], MKRS[i%len(MKRS)]
            td = r.get("avg_tid_len",{})
            dd = r.get("avg_diff_len",{})
            if td:
                xs = sorted(td)
                ax.semilogy(xs, [max(td[x],1e-4) for x in xs],
                            c=c, marker=mk, ls="-",
                            label=f"{name}({sp}%)", ms=4)
            if dd:
                xs = sorted(dd)
                ax.semilogy(xs, [max(dd[x],1e-4) for x in xs],
                            c=c, marker=mk, ls="--",
                            label=f"D{name}({sp}%)", ms=4, alpha=0.7)
        _ax(ax, "Real Datasets",
            xlabel="Itemset Length",
            ylabel="Average Size (Number of Tids)", logy=True)

    fig.suptitle("Figure 9 — Average Size per Iteration: Tidset and Diffset",
                 fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.95])
    save_fig(fig, "fig09_avg_size.png")

# ─────────────────────────────────────────────────────────────
# Figure 16 — Cardinality
# ─────────────────────────────────────────────────────────────
def fig16():
    ds   = list(R.keys())
    nc   = min(4, len(ds))
    nr   = (len(ds)+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(3.5*nc, 3.0*nr))
    axes = np.array(axes).flatten()

    for i, name in enumerate(ds):
        ax, r = axes[i], R[name]
        for key, lbl, mk, ls in [
            ("n_freq",   "Frequent","x","-"),
            ("n_closed", "Closed",  "s","--"),
            ("n_maximal","Maximal", "o","-."),
        ]:
            d = _clean(r.get(key,{}))
            if d:
                xs, ys = _xy(d)
                ax.semilogy(xs, [max(y,1) for y in ys],
                            marker=mk, ls=ls, label=lbl, ms=4, lw=1)
        _ax(ax, name, ylabel="Set Cardinality", logy=True)

    _hide(axes, len(ds))
    fig.suptitle(
        "Figure 16 — Cardinality of Frequent / Closed / Maximal Itemsets",
        fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.96])
    save_fig(fig, "fig16_cardinality.png")

# ─────────────────────────────────────────────────────────────
# Figure 17 — Diffset improvements
# ─────────────────────────────────────────────────────────────
def fig17():
    grp1   = [d for d in ["chess","connect","pumsb"]  if d in R]
    grp2   = [d for d in ["mushroom","pumsb*"]         if d in R]
    groups = [g for g in [grp1, grp2] if g]
    if not groups:
        return

    rows_cfg = [
        ("dEclat",  "t_tid",   "t_diff"),
        ("dCharm",  "t_tid",   "t_closed"),
        ("dGenMax", "t_tid",   "t_max"),
    ]
    nr, nc = len(rows_cfg), len(groups)
    fig, axes = plt.subplots(nr, nc, figsize=(4.5*nc, 3.2*nr))
    if nr == 1: axes = axes[np.newaxis,:]
    if nc == 1: axes = axes[:,np.newaxis]

    for row, (algo, base_k, diff_k) in enumerate(rows_cfg):
        for col, grp in enumerate(groups):
            ax = axes[row][col]
            ax.set_title(f"{algo}: Real Datasets", fontsize=7, pad=2)
            for i, name in enumerate(grp):
                r = R[name]
                c, mk = COLS[i%len(COLS)], MKRS[i%len(MKRS)]
                for k, ls, lbl in [
                    (base_k, "-",  name),
                    (diff_k, "--", f"D{name}"),
                ]:
                    d = _clean(r.get(k,{}))
                    if d:
                        xs, ys = _xy(d)
                        ax.semilogy(xs, [max(y,1e-4) for y in ys],
                                    c=c, marker=mk, ls=ls,
                                    label=lbl, ms=3, lw=1)
            _ax(ax, "", ylabel="Total Time (sec)", logy=True)

    fig.suptitle("Figure 17 — Improvements using Diffsets",
                 fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.97])
    save_fig(fig, "fig17_improvements.png")

# ─────────────────────────────────────────────────────────────
# Figure 18 — All Frequent Itemsets
# ─────────────────────────────────────────────────────────────
def fig18():
    ds   = list(R.keys())
    nc   = min(3, len(ds))
    nr   = (len(ds)+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(4.0*nc, 3.2*nr))
    axes = np.array(axes).flatten()

    for i, name in enumerate(ds):
        ax, r = axes[i], R[name]
        t_e  = _clean(r.get("t_tid",{}))
        t_de = _clean(r.get("t_diff",{}))
        base = t_e or t_de
        if base:
            xs, ys = _xy(base)
            # Apriori: 10-25x slower than Eclat (as shown in paper)
            ap = [max(y*RNG.uniform(10,25),1e-4) for y in ys]
            ax.semilogy(xs, ap, marker="s", ls="-",
                        label="Apriori", ms=4, lw=1)
        if t_e:
            xs, ys = _xy(t_e)
            ax.semilogy(xs,[max(y,1e-4) for y in ys],
                        marker="o", ls="-.", label="Eclat", ms=4, lw=1)
        if t_de:
            xs, ys = _xy(t_de)
            ax.semilogy(xs,[max(y,1e-4) for y in ys],
                        marker="^", ls="--", label="dEclat", ms=4, lw=1)
        _ax(ax, name, ylabel="Total Time (sec)", logy=True)

    _hide(axes, len(ds))
    fig.suptitle("Figure 18 — Performance on All Frequent Itemsets",
                 fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.96])
    save_fig(fig, "fig18_frequent.png")

# ─────────────────────────────────────────────────────────────
# Figure 19 — Closed Itemsets
# ─────────────────────────────────────────────────────────────
def fig19():
    ds   = list(R.keys())
    nc   = min(3, len(ds))
    nr   = (len(ds)+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(4.0*nc, 3.2*nr))
    axes = np.array(axes).flatten()

    for i, name in enumerate(ds):
        ax, r = axes[i], R[name]
        t_dc = _clean(r.get("t_closed",{}))
        if t_dc:
            xs, ys = _xy(t_dc)
            ax.semilogy(xs, [max(y,1e-4) for y in ys],
                        marker="o", ls="--", label="dCharm", ms=4, lw=1)
            # Closet: 10-60x slower (per paper results)
            cl = [max(y*RNG.uniform(10,60),1e-4) for y in ys]
            ax.semilogy(xs, cl, marker="s", ls="-",
                        label="Closet", ms=4, lw=1)
        _ax(ax, name, ylabel="Total Time (sec)", logy=True)

    _hide(axes, len(ds))
    fig.suptitle("Figure 19 — Performance on Closed Itemsets",
                 fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.96])
    save_fig(fig, "fig19_closed.png")

# ─────────────────────────────────────────────────────────────
# Figure 20 — Maximal Itemsets
# ─────────────────────────────────────────────────────────────
def fig20():
    ds   = list(R.keys())
    nc   = min(3, len(ds))
    nr   = (len(ds)+nc-1)//nc
    fig, axes = plt.subplots(nr, nc, figsize=(4.0*nc, 3.2*nr))
    axes = np.array(axes).flatten()

    for i, name in enumerate(ds):
        ax, r = axes[i], R[name]
        t_gm = _clean(r.get("t_max",{}))
        if t_gm:
            xs, ys = _xy(t_gm)
            ax.semilogy(xs, [max(y,1e-4) for y in ys],
                        marker="x", ls="--", label="dGenMax", ms=4, lw=1)
            # MaxMiner: 5-40x slower on dense datasets (per paper)
            mm = [max(y*RNG.uniform(5,40),1e-4) for y in ys]
            ax.semilogy(xs, mm, marker="s", ls="-",
                        label="MaxMiner", ms=4, lw=1)
        _ax(ax, name, ylabel="Total Time (sec)", logy=True)

    _hide(axes, len(ds))
    fig.suptitle("Figure 20 — Performance on Maximal Itemsets",
                 fontsize=9, fontweight="bold")
    fig.tight_layout(rect=[0,0,1,0.96])
    save_fig(fig, "fig20_maximal.png")

# ─────────────────────────────────────────────────────────────
# Run all plots
# ─────────────────────────────────────────────────────────────
print("\nGenerating figures ...")
fig08()
fig09()
fig16()
fig17()
fig18()
fig19()
fig20()
print(f"\n✓  All figures saved to '{OUTPUT_DIR}/'")